# Detecting Label Errors with Cleanlab

This notebook demonstrates how to:
1. Load a public dataset (20 Newsgroups)
2. Purposefully corrupt some labels
3. Use Cleanlab to detect the corrupted labels

**Install requirements:**
```bash
pip install cleanlab scikit-learn pandas numpy matplotlib
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from cleanlab import Datalab

np.random.seed(42)

/Users/nipun/.uv/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load 20 Newsgroups Dataset

We'll use a subset: `rec.sport.baseball` vs `rec.sport.hockey` (binary classification)

In [2]:
# Load only 2 categories for simplicity
categories = ['rec.sport.baseball', 'rec.sport.hockey']
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

texts = newsgroups.data
true_labels = newsgroups.target  # 0 = baseball, 1 = hockey

print(f"Dataset size: {len(texts)} documents")
print(f"Classes: {newsgroups.target_names}")
print(f"Class distribution: {np.bincount(true_labels)}")

Dataset size: 1197 documents
Classes: ['rec.sport.baseball', 'rec.sport.hockey']
Class distribution: [597 600]


In [3]:
# Show a few examples
for i in range(3):
    print(f"\n--- Example {i+1} (Label: {newsgroups.target_names[true_labels[i]]}) ---")
    print(texts[i][:300] + "...")


--- Example 1 (Label: rec.sport.baseball) ---


The tribe will be in town from April 16 to the 19th.
There are ALWAYS tickets available! (Though they are playing Toronto,
and many Toronto fans make the trip to Cleveland as it is easier to
get tickets in Cleveland than in Toronto.  Either way, I seriously
doubt they will sell out until the end o...

--- Example 2 (Label: rec.sport.hockey) ---
This game would have been great as part of a double-header on ABC or
ESPN; the league would have been able to push back-to-back wins by
Le Magnifique and The Great One.  Unfortunately, the only network
that would have done that was SCA, seen in few areas and hard to
justify as a pay channel. )-;

gl...

--- Example 3 (Label: rec.sport.baseball) ---
My god, hope we don't have to put up with this kind of junk all season!


How many home runs by Tartabull?  Just 1, right, you must be thinking
of Dean Palmer or Juan Gonzalez (both of Texas) who each had 2 homers.


I don't know how many to follow, but

## Step 2: Purposefully Corrupt Labels

We'll flip 15% of labels randomly to simulate annotation errors.

In [4]:
# Create noisy labels by flipping 15% randomly
NOISE_RATE = 0.15

noisy_labels = true_labels.copy()
n_to_flip = int(len(noisy_labels) * NOISE_RATE)

# Randomly select indices to flip
flip_indices = np.random.choice(len(noisy_labels), size=n_to_flip, replace=False)
flip_indices = set(flip_indices)

# Flip the labels
for idx in flip_indices:
    noisy_labels[idx] = 1 - noisy_labels[idx]  # 0 -> 1 or 1 -> 0

print(f"Flipped {n_to_flip} labels ({NOISE_RATE:.0%} noise rate)")
print(f"Corrupted indices (first 20): {sorted(list(flip_indices))[:20]}...")

Flipped 179 labels (15% noise rate)
Corrupted indices (first 20): [np.int64(23), np.int64(31), np.int64(43), np.int64(44), np.int64(49), np.int64(51), np.int64(54), np.int64(56), np.int64(58), np.int64(70), np.int64(78), np.int64(81), np.int64(86), np.int64(88), np.int64(96), np.int64(101), np.int64(107), np.int64(109), np.int64(113), np.int64(123)]...


In [5]:
# Verify corruption
n_different = np.sum(true_labels != noisy_labels)
print(f"Labels that differ from ground truth: {n_different}")
print(f"Actual noise rate: {n_different / len(true_labels):.1%}")

Labels that differ from ground truth: 179
Actual noise rate: 15.0%


## Step 3: Create Features and Train Model

In [6]:
# Vectorize text
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(texts)

print(f"Feature matrix shape: {X.shape}")

Feature matrix shape: (1197, 1000)


In [7]:
# Get cross-validated predicted probabilities
# This is important - we need out-of-fold predictions to avoid overfitting

clf = LogisticRegression(max_iter=1000, random_state=42)
pred_probs = cross_val_predict(clf, X, noisy_labels, cv=5, method='predict_proba')

print(f"Predicted probabilities shape: {pred_probs.shape}")
print(f"\nFirst 5 predictions:")
for i in range(5):
    print(f"  [{i}] P(baseball)={pred_probs[i,0]:.2f}, P(hockey)={pred_probs[i,1]:.2f}, Label={noisy_labels[i]}")

Predicted probabilities shape: (1197, 2)

First 5 predictions:
  [0] P(baseball)=0.54, P(hockey)=0.46, Label=0
  [1] P(baseball)=0.23, P(hockey)=0.77, Label=1
  [2] P(baseball)=0.64, P(hockey)=0.36, Label=0
  [3] P(baseball)=0.42, P(hockey)=0.58, Label=1
  [4] P(baseball)=0.72, P(hockey)=0.28, Label=0


## Step 4: Use Cleanlab to Find Label Issues

In [8]:
# Create Datalab and find issues
lab = Datalab(
    data={"text": texts, "label": noisy_labels},
    label_name="label"
)

lab.find_issues(pred_probs=pred_probs)

# Get summary
print(lab.get_issue_summary())

Finding label issues ...
Finding outlier issues ...
Fitting OOD estimator based on provided pred_probs ...
Finding non_iid issues ...
Finding class_imbalance issues ...

Audit complete. 466 issues found in the dataset.
        issue_type         score  num_issues
0            label  8.705096e-01         229
1          outlier  8.481414e-02         236
2          non_iid  6.593392e-14           1
3  class_imbalance  4.970760e-01           0


In [ ]:
# Get detailed issues
issues = lab.get_issues()
label_issues = issues[issues['is_label_issue'] == True]

print(f"\nCleanlab found {len(label_issues)} potential label errors")
print(f"We actually corrupted {n_to_flip} labels")

In [ ]:
# Show some detected issues
print("\nSample detected issues:")
print("-" * 80)

for idx in label_issues.index[:10]:
    text_preview = texts[idx][:80].replace('\n', ' ')
    given = newsgroups.target_names[noisy_labels[idx]]
    predicted = newsgroups.target_names[1 if pred_probs[idx, 1] > 0.5 else 0]
    confidence = max(pred_probs[idx])
    actually_wrong = idx in flip_indices
    
    print(f"[{idx}] Given: {given:10} | Model: {predicted:10} ({confidence:.0%}) | Actually wrong: {actually_wrong}")
    print(f"     \"{text_preview}...\"")
    print()

## Step 5: Evaluate Cleanlab's Performance

In [ ]:
# Compare detected issues with actual corrupted labels
detected_indices = set(label_issues.index)

true_positives = len(detected_indices & flip_indices)
false_positives = len(detected_indices - flip_indices)
false_negatives = len(flip_indices - detected_indices)

precision = true_positives / len(detected_indices) if detected_indices else 0
recall = true_positives / len(flip_indices) if flip_indices else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Cleanlab Performance:")
print("=" * 40)
print(f"True Positives:  {true_positives:4d} (correctly found errors)")
print(f"False Positives: {false_positives:4d} (flagged but actually correct)")
print(f"False Negatives: {false_negatives:4d} (missed errors)")
print(f"\nPrecision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1 Score:  {f1:.1%}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Venn-like comparison
ax = axes[0]
categories = ['Found by\nCleanlab', 'Actually\nCorrupted', 'Both']
values = [len(detected_indices), len(flip_indices), true_positives]
colors = ['steelblue', 'coral', 'green']
ax.bar(categories, values, color=colors)
ax.set_ylabel('Number of examples')
ax.set_title('Cleanlab Detection Results')
for i, v in enumerate(values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')

# Confusion-style breakdown
ax = axes[1]
breakdown = ['True Pos\n(Found errors)', 'False Pos\n(False alarms)', 'False Neg\n(Missed errors)']
values = [true_positives, false_positives, false_negatives]
colors = ['green', 'orange', 'red']
ax.bar(breakdown, values, color=colors)
ax.set_ylabel('Number of examples')
ax.set_title('Detection Breakdown')
for i, v in enumerate(values):
    ax.text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 6: Clean the Dataset

In [ ]:
# Option 1: Remove detected issues
clean_mask = ~issues['is_label_issue']
X_clean = X[clean_mask]
y_clean = noisy_labels[clean_mask]

print(f"Original dataset: {X.shape[0]} examples")
print(f"After removing issues: {X_clean.shape[0]} examples")
print(f"Removed: {X.shape[0] - X_clean.shape[0]} examples")

In [ ]:
# Compare model accuracy: noisy vs clean
from sklearn.model_selection import cross_val_score

# Train on noisy data
scores_noisy = cross_val_score(clf, X, noisy_labels, cv=5)
print(f"Accuracy with noisy labels: {scores_noisy.mean():.1%} (+/- {scores_noisy.std():.1%})")

# Train on cleaned data
scores_clean = cross_val_score(clf, X_clean, y_clean, cv=5)
print(f"Accuracy with clean labels: {scores_clean.mean():.1%} (+/- {scores_clean.std():.1%})")

# Train on true labels (oracle)
scores_true = cross_val_score(clf, X, true_labels, cv=5)
print(f"Accuracy with true labels:  {scores_true.mean():.1%} (+/- {scores_true.std():.1%})")

## Summary

**What we learned:**
1. Real datasets often have label noise (5-20%)
2. Cleanlab can automatically detect many mislabeled examples
3. Removing/fixing noisy labels improves model performance

**Key insight:** Train a model → find where it strongly disagrees with labels → those are likely errors!